# Robotic warehouse (RWARE)

> RWARE (multi-robot warehouse) as a stable-marl environment, with vector and pixel observations.

In [ ]:
#| default_exp envs.rware.env

In [ ]:
#| hide
from nbdev.showdoc import *

The multi-robot warehouse ([RWARE](https://github.com/semitable/robotic-warehouse), Christianos et al.,
MIT license): robots pick up requested shelves, deliver them to the goal cells (bottom row) and return
them to free shelf locations; each delivery is rewarded. It is a stable-marl `MultiAgentEnv` around
RWARE's `Warehouse` (vendored unchanged in `stable_marl.envs.rware.upstream`, see its README), with both
kinds of observations:

| key | per agent | content |
|---|---|---|
| `vector` | `(dim,)` float32 | RWARE's own (flattened) observation of the `sensor_range` square around the agent |
| `pixels` | `(V * tile_size, V * tile_size, 3)` uint8 | an RGB render of the same square (`V = 2 * sensor_range + 1`), world-aligned, outside the warehouse black |
| `direction`, `position`, `carrying` | | heading (the library's convention: 0 right, 1 down, 2 left, 3 up), cell `(x, y)`, whether it carries a shelf |

`observations` chooses `vector`, `pixels` or both (default). The RGB images are drawn here with numpy (in
RWARE's colours), as RWARE's renderer needs a display (pyglet / OpenGL): `render()` gives the whole
warehouse.

Differences from RWARE's gymnasium env: everything is keyed by agent id; RWARE's end of episode (its
`max_steps`, or `max_inactivity_steps` without delivery), which it reports as `done`, is a truncation
of every agent (no agent ever terminates: the task goes on); `state()` is the whole warehouse as
layers; `noop_action` is RWARE's `NOOP`.

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Sequence

import numpy as np
from gymnasium import spaces

from stable_marl.envs.base import MultiAgentEnv
from stable_marl.types import AgentID

### Rendering

Each cell is a `tile_size` square in RWARE's colours: white floor (goal cells dark grey), shelves (teal
when requested), robots as orange discs (red when carrying a shelf) with a black line towards their
heading, and black grid lines.

In [ ]:
#| export
_FLOOR, _GRID, _GOAL = (255, 255, 255), (0, 0, 0), (60, 60, 60)
_SHELF, _SHELF_REQUESTED = (72, 61, 139), (0, 128, 128)
_AGENT, _AGENT_LOADED, _AGENT_DIR = (255, 140, 0), (255, 0, 0), (0, 0, 0)
_OUTSIDE = (0, 0, 0)
_DIR_VEC = {0: (1, 0), 1: (0, 1), 2: (-1, 0), 3: (0, -1)}   # library directions: right, down, left, up


def _tile(tile_size: int, goal: bool, shelf: int, agent: tuple | None) -> np.ndarray:
    """
    One cell: `shelf` 0 (none), 1 (shelf) or 2 (requested shelf); `agent` None or
    ``(direction, loaded)``.
    """
    T = tile_size
    img = np.empty((T, T, 3), np.uint8)
    img[:] = _GOAL if goal else _FLOOR
    if shelf:
        pad = max(1, T // 8)
        img[pad:T - pad, pad:T - pad] = _SHELF_REQUESTED if shelf == 2 else _SHELF
    if agent is not None:
        direction, loaded = agent
        yy, xx = np.mgrid[:T, :T] + 0.5
        c = T / 2
        img[(xx - c) ** 2 + (yy - c) ** 2 <= (0.38 * T) ** 2] = _AGENT_LOADED if loaded else _AGENT
        dx, dy = _DIR_VEC[direction]
        half = max(1, T // 10)
        for s in range(int(c), int(c + 0.45 * T)):                    # heading: centre to the edge
            x, y = int(c + dx * (s - c)), int(c + dy * (s - c))
            img[max(0, y - half):y + half, max(0, x - half):x + half] = _AGENT_DIR
    img[0, :] = _GRID
    img[:, 0] = _GRID
    return img


class _Renderer:
    "Draws windows of the warehouse with cached tiles."
    def __init__(self, tile_size: int):
        self.tile_size, self._cache = tile_size, {}

    def tile(self, *key) -> np.ndarray:
        if key not in self._cache:
            self._cache[key] = _tile(self.tile_size, *key)
        return self._cache[key]

    def window(self, cells: dict, x0: int, y0: int, w: int, h: int, width: int, height: int) -> np.ndarray:
        "The cells ``[x0, x0 + w) x [y0, y0 + h)`` (outside the warehouse: black)."
        T = self.tile_size
        img = np.empty((h * T, w * T, 3), np.uint8)
        img[:] = _OUTSIDE
        for j in range(h):
            for i in range(w):
                x, y = x0 + i, y0 + j
                if 0 <= x < width and 0 <= y < height:
                    img[j * T:(j + 1) * T, i * T:(i + 1) * T] = self.tile(*cells[x, y])
        return img

### The environment

In [ ]:
#| export
_SIZES = {'tiny': (1, 3), 'small': (2, 3), 'medium': (2, 5), 'large': (3, 5)}   # RWARE's (shelf rows, shelf columns)
_DIFFICULTY = {'easy': 2.0, 'normal': 1.0, 'hard': 0.5}                          # requested shelves per agent
_FROM_RWARE_DIR = {0: 3, 1: 1, 2: 2, 3: 0}                                       # RWARE's up, down, left, right
OBSERVATIONS = ('vector', 'pixels')


class RwareEnv(MultiAgentEnv):
    """
    RWARE as a stable-marl env.

    Parameters
    ----------
    size : str
        'tiny' (10 x 11), 'small' (10 x 20), 'medium' (16 x 20) or 'large' (16 x 29), RWARE's presets
    agents : int
        Number of robots
    difficulty : str
        Requested shelves per robot: 'easy' (2), 'normal' (1), 'hard' (0.5), as RWARE's ids
    request_queue_size : int, optional
        Number of requested shelves (overrides `difficulty`)
    sensor_range : int
        Radius of the square each robot observes
    max_steps : int
        Step limit (truncation)
    max_inactivity_steps : int, optional
        Truncation after this many steps without a delivery
    reward_type : str
        'individual' (the delivering robot), 'global' (every robot) or 'two_stage', as in RWARE
    layout : str, optional
        A custom warehouse (RWARE's format: ``x`` shelves, ``g`` goals, ``.`` corridors)
    column_height : int
        Height of the shelf columns (RWARE's ids: 8)
    observations : tuple of str
        ``vector`` and / or ``pixels``
    tile_size : int
        Pixels per cell of the images
    render_mode : str, optional
        'rgb_array' for :meth:`render`
    """
    metadata = {'render_modes': ['rgb_array'], 'render_fps': 10}
    noop_action = 0

    def __init__(self, size: str = 'tiny', agents: int = 2, difficulty: str = 'normal',
                 request_queue_size: int | None = None, sensor_range: int = 1, max_steps: int = 500,
                 max_inactivity_steps: int | None = None, reward_type: str = 'individual', layout: str | None = None,
                 column_height: int = 8, observations: Sequence[str] = OBSERVATIONS, tile_size: int = 8,
                 render_mode: str | None = None):
        from stable_marl.envs.rware.upstream.warehouse import ObservationType, RewardType, Warehouse
        if size not in _SIZES:
            raise ValueError(f"size must be one of {list(_SIZES)}, got {size!r}")
        if difficulty not in _DIFFICULTY:
            raise ValueError(f"difficulty must be one of {list(_DIFFICULTY)}, got {difficulty!r}")
        self.observations = tuple(observations)
        if not self.observations or set(self.observations) - set(OBSERVATIONS):
            raise ValueError(f"observations must be among {OBSERVATIONS}, got {observations!r}")
        rows, columns = _SIZES[size]
        self.num_agents, self.sensor_range, self.tile_size = agents, sensor_range, tile_size
        self.render_mode = render_mode
        self.warehouse = Warehouse(
            shelf_columns=columns, column_height=column_height, shelf_rows=rows, n_agents=agents, msg_bits=0,
            sensor_range=sensor_range, request_queue_size=request_queue_size or max(1, int(agents * _DIFFICULTY[difficulty])),
            max_inactivity_steps=max_inactivity_steps, max_steps=max_steps, reward_type=RewardType[reward_type.upper()],
            layout=layout, observation_type=ObservationType.FLATTENED, render_mode='rgb_array')
        self.height, self.width = self.warehouse.grid_size
        self._renderer = _Renderer(tile_size)
        V = 2 * sensor_range + 1
        agent_space = {
            'direction': spaces.Discrete(4),
            'position': spaces.Box(0, max(self.width, self.height) - 1, (2,), np.int64),
            'carrying': spaces.Discrete(2),
        }
        if 'vector' in self.observations:
            agent_space['vector'] = spaces.Box(-np.inf, np.inf, self.warehouse.observation_space[0].shape, np.float32)
        if 'pixels' in self.observations:
            agent_space['pixels'] = spaces.Box(0, 255, (V * tile_size, V * tile_size, 3), np.uint8)
        self.observation_space = spaces.Dict({a: spaces.Dict(agent_space) for a in range(agents)})
        self.action_space = spaces.Dict({a: spaces.Discrete(5) for a in range(agents)})   # NOOP, FORWARD, LEFT, RIGHT, TOGGLE_LOAD

    @property
    def state_space(self) -> spaces.Space:
        "Layers (height, width, 7): shelf, requested shelf, robot, robot heading + 1, carrying, goal, corridor."
        return spaces.Box(0, 4, (self.height, self.width, 7), np.uint8)

    def state(self) -> np.ndarray:
        w = self.warehouse
        s = np.zeros((self.height, self.width, 7), np.uint8)
        for shelf in w.shelfs:
            s[shelf.y, shelf.x, 0] = 1
        for shelf in w.request_queue:
            s[shelf.y, shelf.x, 1] = 1
        for ag in w.agents:
            s[ag.y, ag.x, 2] = 1
            s[ag.y, ag.x, 3] = _FROM_RWARE_DIR[ag.dir.value] + 1
            s[ag.y, ag.x, 4] = ag.carrying_shelf is not None
        for x, y in w.goals:
            s[y, x, 5] = 1
        s[..., 6] = w.highways
        return s

    def _cells(self) -> dict:
        "Each cell's drawing key: (goal, shelf, agent)."
        w = self.warehouse
        requested = {id(s) for s in w.request_queue}
        shelves = {(s.x, s.y): 2 if id(s) in requested else 1 for s in w.shelfs}
        agents = {(a.x, a.y): (_FROM_RWARE_DIR[a.dir.value], a.carrying_shelf is not None) for a in w.agents}
        goals = set(map(tuple, w.goals))
        return {(x, y): ((x, y) in goals, shelves.get((x, y), 0), agents.get((x, y)))
                for x in range(self.width) for y in range(self.height)}

    def _observations(self, vectors) -> dict[AgentID, dict]:
        w, r = self.warehouse, self.sensor_range
        cells = self._cells() if 'pixels' in self.observations else None
        obs = {}
        for a, ag in enumerate(w.agents):
            o = {'direction': _FROM_RWARE_DIR[ag.dir.value], 'position': np.array([ag.x, ag.y], np.int64),
                 'carrying': int(ag.carrying_shelf is not None)}
            if 'vector' in self.observations:
                o['vector'] = np.asarray(vectors[a], np.float32)
            if cells is not None:
                o['pixels'] = self._renderer.window(cells, ag.x - r, ag.y - r, 2 * r + 1, 2 * r + 1, self.width, self.height)
            obs[a] = o
        return obs

    def reset(self, seed: int | None = None, options: dict | None = None):
        super().reset(seed=seed)
        vectors, _ = self.warehouse.reset(seed=seed, options=options)
        return self._observations(vectors), {a: {} for a in range(self.num_agents)}

    def step(self, actions: dict[AgentID, Any]):
        vectors, rewards, done, truncated, _ = self.warehouse.step([int(actions[a]) for a in range(self.num_agents)])
        end = bool(done or truncated)                     # RWARE's step / inactivity limit: a truncation
        agents = range(self.num_agents)
        return (self._observations(vectors), {a: float(rewards[a]) for a in agents}, {a: False for a in agents},
                {a: end for a in agents}, {a: {} for a in agents})

    def render(self) -> np.ndarray | None:
        "The whole warehouse as an RGB image (``render_mode='rgb_array'``)."
        if self.render_mode != 'rgb_array':
            return None
        return self._renderer.window(self._cells(), 0, 0, self.width, self.height, self.width, self.height)

### Tests

In [ ]:
from fastcore.test import test_fail

env = RwareEnv('tiny', agents=2, render_mode='rgb_array')
obs, infos = env.reset(seed=0)
assert (env.height, env.width) == (11, 10) and set(obs) == {0, 1} and infos == {0: {}, 1: {}}
assert obs[0]['vector'].shape == env.warehouse.observation_space[0].shape and obs[0]['pixels'].shape == (24, 24, 3)
assert env.observation_space[0].contains(obs[0]) and env.state().shape == (11, 10, 7)
assert env.render().shape == (88, 80, 3)

# the same as RWARE: its vector observation, positions, and headings in the library's convention
for a, ag in enumerate(env.warehouse.agents):
    assert list(obs[a]['position']) == [ag.x, ag.y] and obs[a]['carrying'] == 0
    assert obs[a]['direction'] == {'UP': 3, 'DOWN': 1, 'LEFT': 2, 'RIGHT': 0}[ag.dir.name]
twin = RwareEnv('tiny', agents=2)
assert all(np.array_equal(twin.reset(seed=0)[0][a]['vector'], obs[a]['vector']) for a in range(2))   # seeded

# the agent's view: itself in the centre (orange disc), black outside the warehouse
ag = env.warehouse.agents[0]
centre = obs[0]['pixels'][12, 12 - 3]                     # on the disc, off the heading line
assert tuple(centre) in (_AGENT, _AGENT_DIR)
assert tuple(env.render()[ag.y * 8 + 4, ag.x * 8 + 2]) in (_AGENT, _AGENT_DIR)
edge = RwareEnv('tiny', agents=1, sensor_range=2)
edge.reset(seed=0)
edge.warehouse.agents[0].x, edge.warehouse.agents[0].y = 0, 0
o = edge._observations([np.zeros(edge.observation_space[0]['vector'].shape)])[0]
assert (o['pixels'][:16, :] == 0).all() and (o['pixels'][:, :16] == 0).all()   # 2 cells above and left: outside

# steps: no agent terminates; the step limit truncates everyone
env = RwareEnv('tiny', agents=2, max_steps=5)
env.reset(seed=1)
for t in range(5):
    obs, rew, term, trunc, info = env.step({0: 1, 1: 2})
    assert set(rew) == {0, 1} and not any(term.values())
assert all(trunc.values())

# a delivery: robot 0 brought under a requested shelf, loaded, and moved onto a goal cell, is rewarded
env = RwareEnv('tiny', agents=1, request_queue_size=1, max_steps=100)
env.reset(seed=0)
w = env.warehouse
shelf, (gx, gy) = w.request_queue[0], w.goals[0]
robot = w.agents[0]
robot.x, robot.y = shelf.x, shelf.y
w._recalc_grid()
obs, *_ = env.step({0: 4})                                 # TOGGLE_LOAD
assert obs[0]['carrying'] == 1 and env.state()[robot.y, robot.x, 4] == 1
w.grid[1, shelf.y, shelf.x] = 0                            # teleport robot + shelf onto the goal
robot.x, robot.y, shelf.x, shelf.y = gx, gy, gx, gy
w._recalc_grid()
_, rew, *_ = env.step({0: 0})
assert rew[0] == 1.0

# only one kind of observation
assert set(RwareEnv(observations=['vector']).reset(seed=0)[0][0]) == {'vector', 'direction', 'position', 'carrying'}
assert 'vector' not in RwareEnv(observations=['pixels']).reset(seed=0)[0][0]
test_fail(lambda: RwareEnv(observations=['rgb']), contains='observations')
test_fail(lambda: RwareEnv('huge'), contains='size')
# in a World: datasets with both observations, evaluation with a video of the whole warehouse
import tempfile
from pathlib import Path
from stable_marl import World, RandomPolicy
from stable_marl.data import HDF5Dataset
from stable_marl.envs import make
with tempfile.TemporaryDirectory() as tmp:
    world = World('RWARE-Tiny-v0', num_envs=2, agents=2, max_steps=20, add_pixels=True)
    world.set_policy(RandomPolicy(seed=0))
    world.collect(f'{tmp}/rware.h5', episodes=3, seed=0, progress=False)
    ds = HDF5Dataset(path=f'{tmp}/rware.h5', num_steps=4)
    item = ds[0]
    assert item['vector'].shape[:2] == (4, 2) and item['pixels'].shape == (4, 2, 24, 24, 3) and item['action'].shape == (4, 2)
    assert {'render', 'state', 'reward', 'direction', 'position', 'carrying'} <= set(ds.column_names)
    assert list(ds.lengths) == [21, 21, 21]                                  # truncated at max_steps
    res = world.evaluate(episodes=2, seed=0, video=f'{tmp}/videos')
    assert len(list(Path(f'{tmp}/videos').glob('*.mp4'))) == 2 and not res['episode_successes'].any()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()